## READ FILE

In [0]:
path = "/Volumes/lakehouse/adventureworks_multisales/raw_zone/Salesperson.csv"

df = (spark.read
      .option("header", True)
      .option("sep", "\t")
      .csv(path))

df.printSchema()

print("Total rows:", df.count())

print("Preview of data:")
display(df)

## Identifier typing check

In [0]:
from pyspark.sql import functions as F

invalid_keys = df.filter(~F.col("EmployeeKey").rlike("^[0-9]+$"))

print("Check: EmployeeKey should be numeric")
print("Invalid EmployeeKey rows:", invalid_keys.count())

display(invalid_keys)

## Duplicate keys (join reliability)

In [0]:
from pyspark.sql import functions as F

duplicates = (df.groupBy("EmployeeKey")
                .count()
                .filter(F.col("count") > 1))

print("Check: EmployeeKey should be unique")
print("Duplicate keys found:", duplicates.count())

display(duplicates)

## Missing values


In [0]:
from pyspark.sql import functions as F

missing = df.select([
    F.sum((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")).alias(c)
    for c in df.columns
])

print("Check: Missing or blank values per column")
display(missing)

## Text standardization check

In [0]:
from pyspark.sql import functions as F

text_cols = ["Salesperson", "Title", "UPN"]

whitespace = df.select([
    F.sum((F.col(c) != F.trim(F.col(c))).cast("int")).alias(c)
    for c in text_cols
])

print("Check: Leading or trailing whitespace in text fields")
display(whitespace)